# NLP Assignment: Multi-Class Sentiment Analyzer with Error Analysis Using Basic NLP

## 1. Title
**Multi-Class Sentiment Analyzer with Error Analysis Using Basic NLP**

---

## 2. Introduction
Sentiment analysis is a fundamental Natural Language Processing (NLP) task used to identify emotions, attitudes, and opinions in text.

In this project, we build an end-to-end **multi-class sentiment analyzer** that classifies reviews and sentences into three sentiment categories:
1. **Positive**
2. **Neutral**
3. **Negative**

We strictly utilize basic, interpretable NLP and classical machine learning techniques:
- Text Cleaning & Normalization (`re`)
- Term Frequency - Inverse Document Frequency (`TfidfVectorizer`)
- Logistic Regression Classification (`LogisticRegression`)
- Comprehensive Evaluation Metrics (Accuracy, Precision, Recall, F1-Score, Confusion Matrix)
- In-depth **Error Analysis** identifying linguistic failure modes (Negation, Mixed Sentiment, Sarcasm, Neutral Ambiguity, Rare Words, Context Limitations).


## 3. Objective
The primary goals of this project are:
- Implement the complete NLP text classification lifecycle using Python and scikit-learn.
- Train a model to accurately distinguish between Positive, Neutral, and Negative text.
- Rigorously investigate misclassifications through **Error Analysis** to discover *why* the model makes mistakes and document systemic linguistic weaknesses.

---

## 4. Tools and Libraries
The following core libraries are required:
```bash
pip install pandas numpy matplotlib scikit-learn
```


## 5. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

print("All libraries imported successfully!")

## 6. Load Dataset and Exploratory Data Analysis (EDA)
We load `sentiment_data.csv`, inspect its structure, and verify data integrity.

In [ ]:
# Load dataset
df = pd.read_csv("sentiment_data.csv")

# Display first 5 rows
print("First 5 rows:")
display(df.head())

# Shape and column inspection
print("Dataset Shape:", df.shape)
print("Columns:", list(df.columns))

# Check information and missing values
print("
Dataset Info:")
df.info()

print("
Missing values before dropna:")
print(df.isnull().sum())

# Drop missing values if any
df = df.dropna().reset_index(drop=True)
print("
Missing values after dropna:")
print(df.isnull().sum())

## 7. Sentiment Class Distribution and Visualization
Checking whether the dataset is balanced across positive, neutral, and negative classes.

In [ ]:
# Unique classes and counts
print("Sentiment Classes:", df["sentiment"].unique())
print("
Class Value Counts:")
print(df["sentiment"].value_counts())

# Visualize class distribution
plt.figure(figsize=(7, 4))
df["sentiment"].value_counts().plot(kind="bar", color=["#2ecc71", "#3498db", "#e74c3c"])
plt.title("Sentiment Distribution", fontsize=14, fontweight="bold")
plt.xlabel("Sentiment Class", fontsize=12)
plt.ylabel("Number of Samples", fontsize=12)
plt.xticks(rotation=0)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

## 8. Text Preprocessing
Raw natural text contains irrelevant noise such as mixed capitalization, URLs, punctuation, and extraneous spacing. We define a standard NLP normalization function `clean_text`.

In [ ]:
def clean_text(text):
    """Cleans raw text by lowercasing, removing URLs, removing special characters/numbers,
    and collapsing whitespace.
    """
    if not isinstance(text, str):
        return ""
    # 1. Lowercase
    text = text.lower()
    # 2. Remove URLs
    text = re.sub(r"http\S+", "", text)
    # 3. Remove non-alphabetic characters
    text = re.sub(r"[^a-z\s]", "", text)
    # 4. Collapse multiple whitespace characters
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Apply cleaning function
df["clean_text"] = df["text"].apply(clean_text)

# Inspect cleaned text vs original text
df[["text", "clean_text"]].head(8)

## 9. Define Input, Target, and Stratified Train-Test Split
We split the dataset into **80% training data** (to teach the model) and **20% testing data** (to evaluate unseen performance). We use `stratify=y` to preserve exact class proportions in both sets.

In [ ]:
X = df["clean_text"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Total dataset samples : {len(df)}")
print(f"Training set samples  : {len(X_train)} (80%)")
print(f"Testing set samples   : {len(X_test)} (20%)")
print("
Training Class Distribution:")
print(y_train.value_counts(normalize=True).round(3))
print("
Testing Class Distribution:")
print(y_test.value_counts(normalize=True).round(3))

## 10. TF-IDF Feature Extraction
Machine learning algorithms require numerical vectors. We use **TF-IDF (Term Frequency - Inverse Document Frequency)** with:
- `ngram_range=(1, 2)`: Learns both single words (unigrams) and two-word pairs (bigrams like *"not bad"*, *"very boring"*).
- `max_features=5000`: Caps vocabulary size to the most informative tokens.
- We strictly `fit_transform` on the **training set** and only `transform` on the **test set** to prevent data leakage.

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

# Fit and transform training set; transform test set
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Vocabulary size:", len(tfidf.vocabulary_))
print("Training feature matrix shape:", X_train_tfidf.shape)
print("Testing feature matrix shape :", X_test_tfidf.shape)

# Sample learned features
sample_features = list(tfidf.get_feature_names_out())[100:115]
print("
Sample learned n-grams:", sample_features)

## 11. Model Training: Logistic Regression
We train a multi-class **Logistic Regression** classifier with `max_iter=1000` to ensure complete convergence.

In [ ]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train the classifier
model.fit(X_train_tfidf, y_train)
print("Model training complete.")
print("Model classes:", model.classes_)

## 12. Model Evaluation
We evaluate the classifier on the unseen test set using:
1. **Accuracy**: Overall fraction of correct classifications.
2. **Classification Report**: Precision, Recall, and F1-Score per class.
3. **Confusion Matrix**: Visual representation of correct predictions and cross-class errors.

In [ ]:
# Predict on testing data
y_pred = model.predict(X_test_tfidf)

# 1. Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)
")

# 2. Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred, digits=4))

# 3. Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=model.classes_
)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix - Sentiment Classifier", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 13. In-Depth Error Analysis
Error analysis goes beyond accuracy to investigate **why** the model makes mistakes.

We construct an Error DataFrame, calculate error rates, inspect confusion pairs, and dissect specific linguistic error categories:
1. **Negation Errors** (e.g., *"not bad"*, *"never fails"*)
2. **Mixed Sentiment** (e.g., *"acting was good but story was boring"*)
3. **Neutral Sentiment Ambiguity**
4. **Sarcasm**
5. **Rare Vocabulary**
6. **Short Sentences**

In [ ]:
# Create results DataFrame
results = pd.DataFrame({
    "text": X_test.values,
    "actual": y_test.values,
    "predicted": y_pred
})

# Extract incorrect predictions
errors = results[results["actual"] != results["predicted"]].copy()

total_samples = len(results)
incorrect_count = len(errors)
error_rate = incorrect_count / total_samples

print(f"Total test samples      : {total_samples}")
print(f"Incorrect predictions   : {incorrect_count}")
print(f"Overall Error Rate      : {error_rate:.4f} ({error_rate * 100:.2f}%)
")

# Confusion distribution among errors
print("Error Distribution by Actual vs. Predicted:")
print(errors.groupby(["actual", "predicted"]).size())

print("
Sample Misclassified Reviews:")
display(errors.head(15))

### Error Drill-Down: Neutral Class & Linguistic Patterns

In [ ]:
print("--- 1. Neutral Class Errors ---")
neutral_errors = errors[errors["actual"] == "neutral"]
print(f"Total Neutral misclassifications: {len(neutral_errors)}")
display(neutral_errors.head(10))

print("
--- 2. Negation Errors ---")
negation_pattern = r"\b(not|never|no|neither|nor|didnt|wasnt|hardly)\b"
neg_errors = errors[errors["text"].str.contains(negation_pattern, regex=True)]
print(f"Misclassifications containing negation: {len(neg_errors)}")
display(neg_errors.head(5))

print("
--- 3. Mixed Sentiment / Contrast Errors ---")
contrast_pattern = r"\b(but|though|although|however|yet|despite)\b"
mixed_errors = errors[errors["text"].str.contains(contrast_pattern, regex=True)]
print(f"Misclassifications with contrastive clauses: {len(mixed_errors)}")
display(mixed_errors.head(5))

print("
--- 4. Short Sentence Errors (<= 3 words) ---")
short_errors = errors[errors["text"].apply(lambda s: len(s.split()) <= 3)]
print(f"Short sentence misclassifications: {len(short_errors)}")
display(short_errors.head(5))

## 14. Real-Time Sentiment Prediction Function
We build an interactive inference function `predict_sentiment` that preprocesses any novel sentence, vectorizes it, and outputs both the predicted class and confidence probabilities.

In [ ]:
def predict_sentiment(sentence):
    """Preprocesses an arbitrary sentence, extracts TF-IDF features,
    and returns predicted sentiment along with class probabilities.
    """
    cleaned = clean_text(sentence)
    vec = tfidf.transform([cleaned])
    pred = model.predict(vec)[0]
    probs = model.predict_proba(vec)[0]
    prob_dict = {cls: round(prob, 4) for cls, prob in zip(model.classes_, probs)}
    return pred, prob_dict

# Test sentences covering clear sentiment, negations, mixed sentiment, and objective statements
test_cases = [
    "This movie was amazing and fantastic!",
    "The movie was average and ordinary.",
    "I hated the story, it was dreadful and boring.",
    "The acting was excellent.",
    "Nothing special about this movie.",
    "The movie was not bad at all, I actually liked it.",
    "The acting was good but the story was boring.",
    "Great, another boring three-hour movie to waste my Sunday.",
    "The film runs for approximately one hundred and twenty minutes."
]

print(f"{'Input Sentence':<62} | {'Prediction':<10} | {'Class Probabilities'}")
print("-" * 110)
for sent in test_cases:
    pred_label, prob_dict = predict_sentiment(sent)
    prob_summary = f"Neg: {prob_dict.get('negative', 0):.2f}, Neu: {prob_dict.get('neutral', 0):.2f}, Pos: {prob_dict.get('positive', 0):.2f}"
    print(f"{sent[:60]:<62} | {pred_label:<10} | {prob_summary}")

## 15. Summary and Viva Questions & Answers

### Methodology Pipeline
```text
Dataset (sentiment_data.csv)
   ↓
Text Cleaning (clean_text: lowercasing, regex URL/symbol removal)
   ↓
Train-Test Split (80% train, 20% test with stratification)
   ↓
TF-IDF Vectorization (n-grams (1, 2), max_features=5000)
   ↓
Logistic Regression Classifier (max_iter=1000)
   ↓
Model Evaluation (Accuracy, Precision, Recall, F1-Score, Confusion Matrix)
   ↓
Error Analysis (Taxonomy: Negation, Mixed Opinion, Neutral, Sarcasm, Context)
```

---

### Viva Questions & Answers Summary
1. **What is sentiment analysis?**
   An NLP technique to identify the opinion, attitude, or emotion expressed in text.
2. **What classes are used?**
   Three classes: Positive, Neutral, and Negative.
3. **Why is it called multi-class classification?**
   Because there are three target classes rather than two (binary).
4. **Why do we preprocess text?**
   To reduce noise (case differences, punctuation, URLs) and normalize vocabulary for modeling.
5. **What is TF-IDF?**
   Term Frequency - Inverse Document Frequency; it scores words based on how frequently they appear in a document relative to their rarity across all documents.
6. **Why can't Logistic Regression take raw text?**
   Machine learning algorithms operate on numerical feature vectors, not raw strings.
7. **Why do we use an 80/20 train-test split?**
   To provide sufficient data to learn patterns while reserving unseen samples to test generalization.
8. **What does the Confusion Matrix show?**
   A breakdown of actual vs. predicted labels, showing where classes are confused.
9. **Why is error analysis important?**
   Accuracy only tells us how often the model is right; error analysis reveals *why* it fails and guides improvements.
10. **Why are neutral sentences difficult?**
    They lack strong emotional markers and often contain subtle, balanced, or factual language that overlaps with mild positive or negative phrasing.
